In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor, Normalize
from torch import nn 
from torch.nn import functional as F
from tqdm import tqdm
from matplotlib import pyplot as plt

import torch
import numpy as np
import pandas as pd
import os

plt.rcParams['font.family'] = 'SimHei'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
train_ds = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_ds = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=ToTensor()
)

# 将训练集划分为训练集和验证集
train_ds, val_ds = random_split(
    train_ds, [55000, 5000], torch.Generator().manual_seed(42))

In [13]:
subset = train_ds.dataset.data[train_ds.indices].float() / 255.0
mean = subset.mean()
std = subset.std()


print(f"mean: {mean}, std: {std}")


transforms = nn.Sequential(
    Normalize(mean, std)
)

mean: 0.28556573390960693, std: 0.35272860527038574


In [14]:
batch_size = 256
num_workers = 4

train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(
    val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)



In [15]:
class CNN(nn.Module):
    def __init__(self, activation=F.relu):
        super().__init__()
        self.activation = activation

        # 参数说明：
        # in_channels：输入通道数，灰度图像为1，彩色图像为3
        # out_channels：输出通道数，即卷积核的数量
        # kernel_size：卷积核的大小，可以是单个整数（表示宽高相同）或一个元组（表示宽和高）
        # stride：卷积的步幅，默认为1
        # padding：卷积的填充，默认为0，可以是单个整数（表示宽高相同）或一个元组（表示宽和高）

        # 卷积层的设计原则：
        # 1. 卷积层的输出通道数通常是输入通道数的倍数，常见的倍数有2、4、8等，这样可以逐渐增加特征图的数量，提取更多的特征。
        # 2. 卷积核的大小通常选择3x3或5x3，这样可以捕捉局部特征，同时保持计算效率。
        # 3. 卷积层之间可以添加池化层来减少特征图的尺寸，降低计算复杂度，同时增加感受野。

        # 池化层：使用最大池化，池化核大小为2，步幅为2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # 1. 卷积层1：输入通道数为1，输出通道数为32，卷积核大小为3，步幅为1，填充为1
        # (batch_size, 1, 28, 28) -> (batch_size, 32, 28, 28)
        self.conv1 = nn.Conv2d(
            in_channels=1, out_channels=32, kernel_size=3, padding=1)

        # 2. 卷积层2：输入通道数为32，输出通道数为32，卷积核大小为3, 填充为1
        # (batch_size, 32, 28, 28) -> (batch_size, 32, 14, 14)
        self.conv2 = nn.Conv2d(
            in_channels=32, out_channels=32, kernel_size=3, padding=1)

        # 3. 卷积层3：输入通道数为32，输出通道数为64，卷积核大小为3, 填充为1
        # (batch_size, 32, 14, 14) -> (batch_size, 64, 7, 7)
        self.conv3 = nn.Conv2d(
            in_channels=32, out_channels=64, kernel_size=3, padding=1)

        # 4. 卷积层4：输入通道数为64，输出通道数为64，卷积核大小为3, 填充为1
        # (batch_size, 64, 7, 7) -> (batch_size, 64, 3, 3)
        self.conv4 = nn.Conv2d(
            in_channels=64, out_channels=64, kernel_size=3, padding=1)

        # 5. 卷积层5：输入通道数为64，输出通道数为128，卷积核大小为3, 填充为1
        # (batch_size, 64, 3, 3) -> (batch_size, 128, 3, 3)
        self.conv5 = nn.Conv2d(
            in_channels=64, out_channels=128, kernel_size=3, padding=1)

        # 6. 卷积层6：输入通道数为128，输出通道数为128，卷积核大小为3, 填充为1
        # (batch_size, 128, 3, 3) -> (batch_size, 128, 3, 3)
        self.conv6 = nn.Conv2d(
            in_channels=128, out_channels=128, kernel_size=3, padding=1)

        # 7. 展平层：将卷积层的输出展平为一维向量
        self.flatten = nn.Flatten()
        # 8. 全连接层1：输入特征数为128*3*3，输出特征数为128
        self.fc1 = nn.Linear(128 * 3 * 3, 128)
        # 9. 全连接层2：输入特征数为128，输出特征数为10（对应10个类别）
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        act = self.activation
        # 1 * 28 * 28 -> 32 * 14 * 14
        x = self.pool(act(self.conv2(act(self.conv1(x)))))
        # 32 * 14 * 14 -> 64 * 7 * 7
        x = self.pool(act(self.conv4(act(self.conv3(x)))))
        # 64 * 7 * 7 -> 128 * 3 * 3
        x = self.pool(act(self.conv6(act(self.conv5(x)))))

        x = self.flatten(x)
        x = act(self.fc1(x))
        x = self.fc2(x)

        return x


for i, (key, value) in enumerate(CNN().named_parameters()):
    print(f"{key}\tparameter num: {np.prod(value.shape)}")

conv1.weight	parameter num: 288
conv1.bias	parameter num: 32
conv2.weight	parameter num: 9216
conv2.bias	parameter num: 32
conv3.weight	parameter num: 18432
conv3.bias	parameter num: 64
conv4.weight	parameter num: 36864
conv4.bias	parameter num: 64
conv5.weight	parameter num: 73728
conv5.bias	parameter num: 128
conv6.weight	parameter num: 147456
conv6.bias	parameter num: 128
fc1.weight	parameter num: 147456
fc1.bias	parameter num: 128
fc2.weight	parameter num: 1280
fc2.bias	parameter num: 10
